In [4]:
# Section 1: Setup Env

%pip install pyhealth
# %pip install scikit-learn matplotlib seaborn
%pip install pandas numpy torch scikit-learn



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# ==========================================
# Step 1: Load MIMIC-III dataset with PyHealth
# ==========================================

from pyhealth.datasets import MIMIC3Dataset
import pandas as pd

DATA_PATH = "../MIMIC3/raw" # CHANGE THIS TO YOUR FOLDER PATH OF THE UNZIPPED FILES

# Initialize dataset
mimic_dataset = MIMIC3Dataset(
    root="../MIMIC3/raw",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "LABEVENTS", "PRESCRIPTIONS"],
    code_mapping={"ICD9CM": "CCSCM"},
    refresh_cache=True,
    dev=True
)

# Check dataset stats
mimic_dataset.stat()

# Load lab item mappings to identify creatinine ITEMIDs
lab_items_df = pd.read_csv(DATA_PATH + "/D_LABITEMS.csv")
creatinine_itemids = set(lab_items_df[lab_items_df['LABEL'].str.contains("Creatinine", case=False)]["ITEMID"].astype(str))

lab_events_df = pd.read_csv(DATA_PATH + "/LABEVENTS.csv")


INFO: Pandarallel will run on 6 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.
finish basic patient information parsing : 1.2388389110565186s
finish parsing DIAGNOSES_ICD : 0.7801792621612549s
finish parsing PROCEDURES_ICD : 2.3617210388183594s
finish parsing LABEVENTS : 70.29452276229858s
finish parsing PRESCRIPTIONS : 20.58882713317871s


Mapping codes: 100%|██████████| 1000/1000 [00:00<00:00, 1467.84it/s]



Statistics of base dataset (dev=True):
	- Dataset: MIMIC3Dataset
	- Number of patients: 1000
	- Number of visits: 1295
	- Number of visits per patient: 1.2950
	- Number of events per visit in DIAGNOSES_ICD: 9.3544
	- Number of events per visit in PROCEDURES_ICD: 4.3351
	- Number of events per visit in LABEVENTS: 399.7104
	- Number of events per visit in PRESCRIPTIONS: 59.7336



In [6]:
from pyhealth.data import Patient, Event

# Custom AKI definition based on creatinine levels using lab_events_df
def aki_prediction_task_fn(patient: Patient):
    samples = []

    for visit in patient:
        visit_lab_events = visit.get_event_list("LABEVENTS")
        creatinine_levels = []
        for event in visit_lab_events:
            code = event.code
            if code in creatinine_itemids:
                matched_rows = lab_events_df[lab_events_df['ITEMID'].astype(str) == code]
                creatinine_levels.extend(matched_rows['VALUENUM'].tolist())

        # print(creatinine_levels)        
        # creatinine_levels = visit_lab_events[visit_lab_events.code.isin(creatinine_itemids)]['VALUENUM'].dropna().tolist()
        
        # print("C Levels", visit.get_event_list("LABEVENTS"))
        if creatinine_levels:
            max_creatinine = max(creatinine_levels)
            label = 1 if max_creatinine >= 1.5 else 0
            samples.append({
                "visit_id": visit.visit_id,
                "patient_id": patient.patient_id,
                "conditions": visit.get_code_list("conditions"),
                "procedures": visit.get_code_list("procedures"),
                "labs": visit.get_code_list("LABEVENTS"),
                "prescriptions": visit.get_code_list("prescriptions"),
                "label": label
            })
    
    return samples

# Apply custom task to dataset
mimic_sample_dataset = mimic_dataset.set_task(task_fn=aki_prediction_task_fn)

# Split dataset
train_dataset, val_dataset, test_dataset = mimic_sample_dataset.split([0.7, 0.1, 0.2])

print("Train dataset: ", train_dataset)
print("Values dataset: ", val_dataset)
print("Test dataset: ", test_dataset)


Generating samples for aki_prediction_task_fn:  49%|████▉     | 490/1000 [22:25:48<23:20:44, 164.79s/it] 


KeyboardInterrupt: 

In [ ]:

# ==========================================
# Step 3: Define the RNN Model
# ==========================================

from pyhealth.models import RNN
from pyhealth.trainer import Trainer

# Define the RNN model (LSTM-based)
model = RNN(
    dataset=train_dataset,
    feature_keys=["conditions", "procedures", "labs", "prescriptions"],
    label_key="label",
    mode="binary",
    embedding_dim=128,
    hidden_dim=64,
    num_layers=3,
    rnn_type="LSTM"
)

In [ ]:
# ==========================================
# Step 4: Train and Evaluate the Model
# ==========================================

trainer = Trainer(model=model, device="cuda:0")
trainer.train(
    train_dataloader=train_dataset.get_dataloader(batch_size=64, shuffle=True),
    epochs=10,
    val_dataloader=val_dataset.get_dataloader(batch_size=64, shuffle=False),
    monitor="pr_auc",
)

# Evaluate on test set
trainer.evaluate(test_dataset.get_dataloader(batch_size=64))

In [ ]:
# ==========================================
# Step 5: Extract Model Activations for TCAV
# ==========================================

import torch

def extract_activations(model, dataloader):
    activations, labels = [], []
    model.eval()
    with torch.no_grad():
        for data in dataloader:
            inputs = model.prepare_input(data, model.feature_keys)
            output, hidden = model.model(inputs)
            activations.append(hidden[-1].cpu().numpy())  # last LSTM layer hidden state
            labels.append(data["label"].numpy())
    return activations, labels

train_acts, train_labels = extract_activations(model, train_dataset.get_dataloader(batch_size=64))

In [ ]:


# ==========================================
# Step 6: Build Concept Activation Vectors (CAVs)
# ==========================================

from sklearn.linear_model import LogisticRegression
import numpy as np

# Example concept: "NSAID prescription"
def define_concept(dataset, concept_key="NSAID"):
    concept_labels = []
    for patient in dataset.samples:
        prescriptions = patient["prescriptions"]
        concept_labels.append(1 if concept_key in prescriptions else 0)
    return np.array(concept_labels)

concept_labels = define_concept(train_dataset, "NSAID")


# Train CAV classifier
cav_classifier = LogisticRegression(max_iter=1000)
cav_classifier.fit(np.concatenate(train_acts), concept_labels)

In [ ]:



# ==========================================
# Step 7: Evaluate TCAV
# ==========================================

# Calculate concept influence
coefs = cav_classifier.coef_[0]
concept_importance = np.mean(coefs)
print(f"Concept Importance (NSAIDs): {concept_importance}")

In [ ]:
# ==========================================
# Step 8: Local Explanation for Single Patient
# ==========================================

# Choose a single patient from test set
patient_data = test_dataset.samples[0]
patient_dataloader = test_dataset.get_dataloader(batch_size=1, shuffle=False)
patient_acts, _ = extract_activations(model, patient_dataloader)

# Compute concept alignment
alignment_score = np.dot(patient_acts[0], coefs) / (np.linalg.norm(patient_acts[0]) * np.linalg.norm(coefs))
print(f"Alignment score for concept (NSAIDs) for patient 0: {alignment_score}")


In [ ]:

# ==========================================
# Step 9: Next Steps and Extensions
# ==========================================

# - Experiment with more clinical concepts (infection, gender, etc.)
# - Automate and expand TCAV analysis
# - Visualization of alignment and concept sensitivity scores
